# 10장 Creating Text Embedding Models — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter10/Chapter 10 - Creating Text Embedding Models.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter10/Chapter%2010%20-%20Creating%20Text%20Embedding%20Models.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter10` 노트북. 2장·8장에서 **가져다 쓰던** 문장 임베딩 모델을 이번에는 **직접 학습**합니다. 핵심은 대조 학습(contrastive learning)입니다. 비슷한 문장 쌍은 가깝게, 다른 문장 쌍은 멀게 벡터를 조정합니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | 공통 설정: 데이터 크기(빠른 실행 옵션), 평가기, 학습 인자, 결과 기록 | 408–414 |
| 2 | `bert-base-uncased`를 NLI 데이터 + **소프트맥스 손실**로 학습, STSB·MTEB로 평가 | 408–416 |
| 3 | 손실 함수 비교: **코사인 유사도 손실**, **MNR 손실**(+ 배치 내 네거티브 시각화) | 416–425 |
| 4 | 이미 학습된 임베딩 모델 미세조정(supervised fine-tuning) | 426–428 |
| 5 | **Augmented SBERT**: 크로스 인코더로 라벨을 늘려 학습 | 429–434 |
| 6 | 라벨 없이 학습: **TSDAE** | 435–442 |
| 7 | (+) 모든 실험 결과 한 그래프로 비교 | — |

`(+ …)` 표시는 책에 없는 보충 실험입니다.

**권장 환경.** `ml.g4dn.xlarge`(T4 16GB)로 충분합니다. 학습 데이터는 GLUE의 MNLI(자연어 추론), 평가 데이터는 STSB(문장 유사도)입니다. 책 설정(문장 쌍 5만 개)으로는 학습 한 번에 수 분 이상 걸리고, 이 노트북에는 학습이 7번 있습니다. 스터디 시간에는 1절의 `QUICK = True`(데이터 1/10)로 흐름을 먼저 보고, 책과 같은 숫자를 보려면 `False`로 다시 실행하세요. 스토리지는 앞 장 캐시에 더해 5GB 정도면 됩니다(체크포인트 저장을 끔).

**학습이 끝날 때마다** 책은 커널 재시작을 권합니다(GPU 메모리 정리). 이 노트북은 결과를 `ch10_results.json`에 저장하므로, 재시작한 뒤 **1절 공통 설정 셀만 다시 실행**하고 다음 절로 넘어가면 됩니다.

## 0. 환경 점검과 패키지 설치

`sentence-transformers`, `datasets`, `mteb`, `nltk`, `accelerate` 중 없는 것만 설치합니다.

이 노트북은 2026-11 기준 최신 버전(sentence-transformers 6.2, transformers 5.20, datasets 5.2, mteb 2.23)으로 검증했습니다. 버전을 책(sentence-transformers 3.0.1, mteb 1.12)에 고정하지 않은 이유는 SageMaker 이미지의 다른 패키지와 충돌하기 때문입니다. 책 코드와 달라진 부분은 해당 셀에 **책과 다를 수 있는 점**으로 적었습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
need = {"sentence_transformers": "sentence-transformers", "datasets": "datasets", "mteb": "mteb",
        "nltk": "nltk", "accelerate": "accelerate"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("추가 설치 불필요")
import torch, transformers, sentence_transformers, datasets
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__, "| datasets", datasets.__version__)

## 1. 공통 설정 (커널을 재시작했다면 이 셀부터 다시 실행)

> **책과 다를 수 있는 점 (데이터셋 이름).** 책의 `load_dataset("glue", "mnli")`는 최신 `datasets`·`huggingface_hub`에서 "저장소 이름은 namespace/name 형식이어야 한다"는 오류로 실패합니다. 같은 데이터가 `nyu-mll/glue`로 옮겨져 있어 이 이름을 씁니다.

> **책과 다를 수 있는 점 (학습 인자).** 책의 학습 인자 값(에폭 1, 배치 32, warmup 100, fp16, eval/logging 100스텝)은 그대로 두고 세 가지만 더했습니다. `save_strategy="no"`는 체크포인트를 저장하지 않게 합니다. 기본값이면 500스텝마다 모델과 옵티마이저 상태(약 1.3GB)를 저장해 6번 학습하면 스토리지가 수십 GB 찹니다. `report_to="none"`은 외부 실험 추적 도구로 로그를 보내지 않게 합니다. `fp16`은 GPU가 있을 때만 켭니다.

평가기(`EmbeddingSimilarityEvaluator`)는 STSB 검증 문장 쌍 1,500개에 대해, 모델이 낸 코사인 유사도와 사람이 매긴 유사도 점수(0~5를 0~1로 환산)의 **순위 상관(Spearman)**을 계산합니다. 1에 가까울수록 사람 판단과 순서가 같습니다.

In [ ]:
import gc, json, os, random
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

QUICK = False            # True: 데이터 1/10로 빠르게 흐름만 확인 / False: 책과 같은 크기
def N(n): return n // 10 if QUICK else n
GLUE = "nyu-mll/glue"    # 책: "glue"

val_sts = load_dataset(GLUE, "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score / 5 for score in val_sts["label"]],
    main_similarity="cosine",
)

def make_args(output_dir, batch_size=32):
    return SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        warmup_steps=100,
        fp16=torch.cuda.is_available(),
        eval_steps=100,
        logging_steps=100,
        save_strategy="no",      # 추가: 체크포인트 저장 안 함
        report_to="none",        # 추가: 외부 로깅 끔
    )

RESULTS = "ch10_results.json"
def record(name, scores):
    r = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    r[name] = round(float(scores["spearman_cosine"]), 4)
    json.dump(r, open(RESULTS, "w"), indent=1, ensure_ascii=False)
    print(f"{name}: spearman_cosine = {r[name]}  (QUICK={QUICK})")

def cleanup(*names):
    for n in names:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU 사용량: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("QUICK =", QUICK, "| 학습 쌍 수 =", N(50_000))

## 2. 임베딩 모델 만들기: 소프트맥스 손실 (PDF 408–416)

### 2-1. 데이터

MNLI는 (전제, 가설) 문장 쌍과 관계 라벨을 담고 있습니다. 0 = 함의(entailment), 1 = 중립(neutral), 2 = 모순(contradiction)입니다. 앞 5만 쌍만 씁니다.

In [ ]:
train_dataset = load_dataset(GLUE, "mnli", split="train").select(range(N(50_000)))
train_dataset = train_dataset.remove_columns("idx")
train_dataset[2]

### 2-2. 모델과 손실 함수

`SentenceTransformer("bert-base-uncased")`는 BERT 위에 **평균 풀링(mean pooling)** 층을 붙여 문장 벡터 1개(768차원)를 만듭니다. 아직 문장 임베딩용으로 학습되지 않은 상태입니다(출력에 `Pooling(... 'mean' ...)`이 보입니다).

**소프트맥스 손실(SoftmaxLoss)**은 두 문장 벡터 u, v로 [u, v, |u−v|]를 만들어 3가지 관계 중 하나로 분류하게 학습합니다. 분류를 잘하려면 벡터가 문장 의미를 담아야 하므로, 그 부산물로 쓸 만한 임베딩이 생깁니다(SBERT 원논문 방식).

In [ ]:
from sentence_transformers import SentenceTransformer, losses

embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3,
)
embedding_model

> **책과 다를 수 있는 점.** sentence-transformers 6.x에서는 `sentence_transformers.losses` 같은 기존 import 경로와 `sentence_embedding_dimension` 인자 이름에 "deprecated" 경고가 나옵니다. 지금은 그대로 동작합니다. 새 경로는 `sentence_transformers.sentence_transformer.losses`이고, 새 인자 이름은 `embedding_dimension`입니다.

### 2-3. 학습 전 점수와 학습

In [ ]:
before = evaluator(embedding_model)
record("bert-base (학습 전)", before)

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=make_args("base_embedding_model"),
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)
trainer.train()

**관찰 포인트.** 책의 결과는 `spearman_cosine` 약 0.60입니다. 학습 전 점수와 비교해 보세요.

> **책과 다를 수 있는 점.** 책 출력에는 manhattan, euclidean, dot 점수도 있지만, 최신 버전은 `main_similarity`로 지정한 코사인 점수만 돌려줍니다.

In [ ]:
scores = evaluator(embedding_model)
record("softmax (bert-base, NLI)", scores)
scores

### 2-4. MTEB로 평가하기 (PDF 414–416)

MTEB(Massive Text Embedding Benchmark)는 분류·군집화·검색 등 여러 작업으로 임베딩 모델을 비교하는 벤치마크입니다. 책은 은행 고객 문의 77개 의도 분류(Banking77) 하나만 돌립니다. 임베딩 위에 로지스틱 회귀 분류기를 학습해 정확도를 잽니다.

> **책과 다를 수 있는 점.** mteb 2.x는 API가 바뀌었습니다. 책의 `MTEB(tasks=[...]).run(model)`(구 방식, 경고와 함께 아직 동작) 대신 `mteb.get_tasks()` + `mteb.evaluate()`를 씁니다. `cache=None`은 디스크에 저장된 이전 평가 결과를 다시 쓰지 않고 매번 새로 평가하게 합니다. 책 코드의 `evaluation.run(model)`은 앞에서 정의하지 않은 `model`을 넘기는 오타라서 `embedding_model`로 고쳤습니다. 평가 절차가 버전마다 조금씩 달라 책의 정확도(0.49)와 숫자가 다를 수 있습니다.

In [ ]:
import mteb

tasks = mteb.get_tasks(tasks=["Banking77Classification"])
results = mteb.evaluate(embedding_model, tasks=tasks, cache=None)
for tr in results.task_results:
    print(tr.task_name, "main_score(accuracy) =", round(tr.get_score(), 4))

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

## 3. 손실 함수 비교 (PDF 416–425)

같은 데이터와 같은 BERT로 손실 함수만 바꿔 봅니다.

### 3-1. 코사인 유사도 손실 (PDF 417–419)

라벨을 유사도 점수로 바꿉니다. 함의는 1, 중립·모순은 0입니다. 모델은 두 문장 벡터의 코사인 유사도가 이 점수에 가까워지도록 학습합니다(회귀). 책 결과는 약 0.73입니다.

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer

train_dataset = load_dataset(GLUE, "mnli", split="train").select(range(N(50_000)))
train_dataset = train_dataset.remove_columns("idx")
mapping = {2: 0, 1: 0, 0: 1}     # (중립/모순)=0, (함의)=1
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]],
})

embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("cosineloss_embedding_model"),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("cosine loss (bert-base)", scores); scores

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

### 3-2. MNR 손실 (PDF 420–425)

**MNR(Multiple Negatives Ranking) 손실**은 (질문, 정답) 같은 **양성 쌍만** 있으면 됩니다. 한 배치 안에서 다른 쌍의 정답을 모두 오답(네거티브)으로 간주하고, 각 질문이 자기 정답을 가장 높게 고르도록 학습합니다. 책은 여기에 무작위로 고른 가설을 "약한 네거티브(soft negative)"로 하나 더 붙입니다.

> **책과 다를 수 있는 점.** 최신 `datasets`(5.x에서 확인)에서는 `mnli["hypothesis"]`가 리스트가 아니라 `Column` 객체를 돌려줍니다. 그래서 책 코드의 `random.shuffle()`이 "item assignment를 지원하지 않는다"는 오류를 냅니다. `list()`로 감쌌습니다. 책 코드에는 `import random`도 빠져 있어 1절에 넣었습니다.

#### (+ 보충) 배치 안에서 무슨 일이 일어나나

이미 학습된 작은 모델(`all-MiniLM-L6-v2`)로 4쌍짜리 배치를 만들어, MNR이 최적화하는 행렬을 직접 계산합니다. 행 = 질문(anchor), 열 = 정답 후보(positive)입니다. MNR은 각 행에서 **대각선 칸이 1등**이 되도록 하는 4지선다 분류(교차 엔트로피)입니다. sentence-transformers의 기본 배율(scale)은 20입니다.

In [ ]:
import pandas as pd, torch.nn.functional as F
probe = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
anchors   = ["How do I reset my password?", "The car stopped at the red light.",
             "What is the capital of France?", "A man is playing a guitar."]
positives = ["Steps to recover a forgotten password", "The vehicle halted at the traffic signal.",
             "Paris is France's capital city.", "Someone is strumming a guitar."]
a = probe.encode(anchors, convert_to_tensor=True, normalize_embeddings=True)
p = probe.encode(positives, convert_to_tensor=True, normalize_embeddings=True)
sim = a @ p.T
loss = F.cross_entropy(sim * 20, torch.arange(len(anchors), device=sim.device))
display(pd.DataFrame(sim.cpu().numpy(), index=[s[:28] for s in anchors], columns=[s[:22] for s in positives]).round(2))
print(f"MNR loss (scale 20) = {loss.item():.2e}  ← 대각선이 행마다 확실히 1등이면 0에 가깝고, 헷갈리는 행이 있으면 커짐")
print("행별 정답 확률:", (sim * 20).softmax(dim=-1).diag().cpu().numpy().round(4))
del probe

**관찰 포인트.** 배치가 클수록 오답 후보가 많아져 문제가 어려워집니다. 그래서 책의 TIP처럼 MNR은 큰 배치에서 더 잘 학습됩니다. 책 결과는 약 0.81로, 세 손실 중 가장 높습니다.

In [ ]:
from tqdm import tqdm

mnli = load_dataset(GLUE, "mnli", split="train").select(range(N(50_000)))
mnli = mnli.remove_columns("idx")
mnli = mnli.filter(lambda x: True if x["label"] == 0 else False)     # 함의 쌍만 = 양성 쌍

train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = list(mnli["hypothesis"])                             # 책: mnli["hypothesis"]
random.shuffle(soft_negatives)
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)

embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("mnrloss_embedding_model"),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("MNR loss (bert-base)", scores); scores

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

## 4. 이미 학습된 임베딩 모델 미세조정 (PDF 426–428)

처음부터 학습하는 대신 이미 좋은 임베딩 모델(`all-MiniLM-L6-v2`)을 출발점으로 같은 MNR 손실로 더 학습합니다. 책 결과는 약 0.85입니다.

**관찰 포인트.** 책 코드는 라벨 열이 있는 MNLI를 그대로 넣습니다. MNR 손실은 라벨을 쓰지 않으므로 모순·중립 쌍까지 양성 쌍으로 학습됩니다. 3-2처럼 함의 쌍만 골라 학습하면 결과가 어떻게 바뀌는지 직접 비교해 볼 만합니다.

In [ ]:
train_dataset = load_dataset(GLUE, "mnli", split="train").select(range(N(50_000)))
train_dataset = train_dataset.remove_columns("idx")

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
record("all-MiniLM-L6-v2 (원본)", evaluator(embedding_model))

train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("finetuned_embedding_model"),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("MiniLM fine-tuned (MNR)", scores); scores

미세조정 전 원본 점수(바로 위에서 기록)와 비교합니다. `all-MiniLM-L6-v2`는 이미 10억 개가 넘는 문장 쌍으로 학습된 모델이라(모델 카드 기준), MNLI 5만 쌍으로 더 학습해도 이득이 작거나 오히려 내려갈 수 있습니다.

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

## 5. Augmented SBERT: 라벨이 적을 때 (PDF 429–434)

라벨이 붙은 데이터가 조금(gold)밖에 없을 때 쓰는 방법입니다.

1. 정확하지만 느린 **크로스 인코더**(두 문장을 한꺼번에 BERT에 넣어 관계를 분류)를 gold 1만 쌍으로 학습합니다.
2. 라벨 없는 문장 쌍 4만 개를 준비합니다.
3. 크로스 인코더로 그 쌍에 라벨을 붙입니다(silver).
4. gold + silver로 빠른 **바이 인코더**(SBERT, 문장마다 따로 벡터를 만드는 구조)를 학습합니다.

크로스 인코더는 쌍마다 BERT를 한 번씩 돌려야 해서 검색에는 쓰기 어렵습니다. 대신 라벨을 만드는 "선생님" 역할을 맡깁니다.

In [ ]:
import numpy as np
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader

dataset = load_dataset(GLUE, "mnli", split="train").select(range(N(10_000)))
mapping = {2: 0, 1: 0, 0: 1}
gold_examples = [InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
                 for row in tqdm(dataset)]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)
gold = pd.DataFrame({
    "sentence1": dataset["premise"],
    "sentence2": dataset["hypothesis"],
    "label": [mapping[label] for label in dataset["label"]],
})

**1단계: 크로스 인코더 학습.**

> **책과 다를 수 있는 점.** sentence-transformers 4.0부터 크로스 인코더에도 전용 Trainer가 생겼고, 책의 `CrossEncoder.fit()`은 기존 방식으로 남아 있습니다(6.2에서 동작 확인).

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

cross_encoder = CrossEncoder("bert-base-uncased", num_labels=2)
cross_encoder.fit(train_dataloader=gold_dataloader, epochs=1, show_progress_bar=True,
                  warmup_steps=100, use_amp=False)

**2·3단계: 새 문장 쌍에 크로스 인코더로 라벨 붙이기(silver).**

In [ ]:
silver = load_dataset(GLUE, "mnli", split="train").select(range(N(10_000), N(50_000)))
pairs = list(zip(silver["premise"], silver["hypothesis"]))

output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
silver = pd.DataFrame({
    "sentence1": silver["premise"],
    "sentence2": silver["hypothesis"],
    "label": np.argmax(output, axis=1),
})
print("silver 라벨 분포:", silver["label"].value_counts().to_dict())

**4단계: gold + silver로 바이 인코더 학습.** 책 결과는 약 0.72입니다.

> **책과 다를 수 있는 점.** 책 본문은 이 단계에서 `# Define model` 주석만 있고 모델 정의 줄이 빠져 있습니다. 앞 절과 같이 `bert-base-uncased`로 정의했습니다.

In [ ]:
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=["sentence1", "sentence2"], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("augmented_embedding_model"),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("Augmented SBERT (gold+silver)", scores); scores

**5단계: silver 없이 gold만으로 학습해 비교.** 공식 노트북 결론은 "gold만 쓰면 성능이 내려간다"입니다.

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

train_dataset = Dataset.from_pandas(gold, preserve_index=False)
embedding_model = SentenceTransformer("bert-base-uncased")
train_loss = losses.CosineSimilarityLoss(model=embedding_model)
trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("gold_only_embedding_model"),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("gold only (no silver)", scores); scores

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss", "cross_encoder")

## 6. 라벨 없이 학습하기: TSDAE (PDF 435–442)

**TSDAE(Transformer-based Sequential Denoising Auto-Encoder)**는 문장에서 단어를 일부(기본 60%) 지운 "손상된 문장"을 인코더로 벡터 하나에 담고, 디코더가 그 벡터만 보고 **원래 문장을 복원**하게 학습합니다. 벡터 하나로 문장을 복원하려면 그 벡터가 문장 의미를 압축해 담아야 합니다. 라벨이 필요 없어 자기 도메인 문장만 있으면 됩니다(도메인 적응, 441쪽).

> **책과 다를 수 있는 점.** 문장 손상에 쓰는 NLTK 토크나이저는 최신 NLTK(3.8.2 이상)에서 `punkt_tab` 리소스를 찾습니다. 책의 `punkt`와 함께 둘 다 내려받습니다. `mnli["premise"] + mnli["hypothesis"]`도 최신 `datasets`에서는 `Column` 객체끼리 더할 수 없어 `list()`로 감쌌습니다.

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from sentence_transformers.datasets import DenoisingAutoEncoderDataset

mnli = load_dataset(GLUE, "mnli", split="train").select(range(N(25_000)))
flat_sentences = list(mnli["premise"]) + list(mnli["hypothesis"])     # 책: mnli["premise"] + mnli["hypothesis"]
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))

train_dataset = {"damaged_sentence": [], "original_sentence": []}
for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])
train_dataset = Dataset.from_dict(train_dataset)
train_dataset[0]

TSDAE는 평균 풀링 대신 **[CLS] 토큰 풀링**을 씁니다(TSDAE 논문 권장). 디코더는 인코더와 가중치를 공유합니다(`tie_encoder_decoder=True`). 책 결과는 약 0.71로, 라벨 없이도 소프트맥스 손실(0.60)보다 높습니다.

> **책과 다를 수 있는 점.** 책 본문의 Trainer 호출에는 `train_dataset=` 줄이 빠져 있습니다. 공식 노트북처럼 넣었습니다.

In [ ]:
from sentence_transformers import models, SentenceTransformer

word_embedding_model = models.Transformer("bert-base-uncased")
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), "cls")
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

train_loss = losses.DenoisingAutoEncoderLoss(embedding_model, tie_encoder_decoder=True)
train_loss.decoder = train_loss.decoder.to(embedding_model.device)    # 책: .to("cuda")

trainer = SentenceTransformerTrainer(
    model=embedding_model, args=make_args("tsdae_embedding_model", batch_size=16),
    train_dataset=train_dataset, loss=train_loss, evaluator=evaluator,
)
trainer.train()
scores = evaluator(embedding_model); record("TSDAE (unsupervised)", scores); scores

In [ ]:
trainer.accelerator.clear()
cleanup("trainer", "embedding_model", "train_loss")

## 7. (+ 보충) 모든 실험 한눈에 보기

`ch10_results.json`에 쌓인 점수를 그립니다. 중간에 커널을 재시작했어도 기록은 남아 있습니다. 그래프 글자는 영어로 둡니다(Studio 기본 이미지에 한글 폰트가 없을 수 있음).

In [ ]:
import matplotlib.pyplot as plt
r = json.load(open(RESULTS))
book = {"softmax (bert-base, NLI)": 0.603, "cosine loss (bert-base)": 0.725, "MNR loss (bert-base)": 0.812,
        "MiniLM fine-tuned (MNR)": 0.848, "Augmented SBERT (gold+silver)": 0.721, "TSDAE (unsupervised)": 0.714}
df = pd.DataFrame({"this run": r, "book": book}).loc[list(r)]
eng = {"bert-base (학습 전)": "bert-base (untrained)", "all-MiniLM-L6-v2 (원본)": "all-MiniLM-L6-v2 (original)"}
ax = df.rename(index=eng).plot.barh(figsize=(8, 4.5), title=f"STSB spearman_cosine (QUICK={QUICK})")
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.grid(axis="x", alpha=.3)
df

**읽는 법.** "book" 열은 책에 인쇄된 값(spearman_cosine)입니다. `QUICK=True`로 돌렸다면 데이터가 1/10이라 낮게 나오는 것이 정상입니다. 순서를 보세요. 학습 전 BERT < 소프트맥스 < 코사인 < MNR 순으로 오르는지, 그리고 라벨 없는 TSDAE가 어디쯤 오는지가 이 장의 핵심 비교입니다.

## 8. 정리

| 방법 | 필요한 데이터 | 학습 신호 | 책 점수 |
|---|---|---|---|
| 소프트맥스 손실 | 라벨 3종 문장 쌍 | 관계 분류 | 0.60 |
| 코사인 유사도 손실 | 유사도 점수가 붙은 쌍 | 코사인 회귀 | 0.73 |
| MNR 손실 | 양성 쌍만 | 배치 내 순위(대조) | 0.81 |
| 기존 모델 미세조정 | 양성 쌍 | MNR | 0.85 |
| Augmented SBERT | 적은 gold + 라벨 없는 쌍 | 크로스 인코더가 만든 silver 라벨 | 0.72 |
| TSDAE | 문장만 (라벨 없음) | 손상 문장 복원 | 0.71 |

**자율주행·Agent로 이어지는 점(책 밖).** 8장 RAG와 Agent의 도구·문서 검색 품질은 임베딩 모델이 결정합니다. 사내 정비 매뉴얼, 주행 이벤트 리포트, 시나리오 설명처럼 범용 모델이 잘 모르는 도메인이라면, 이 장의 순서대로 해 볼 수 있습니다. 먼저 TSDAE로 도메인 문장에 적응시키고, (질문, 정답 문서) 양성 쌍을 모아 MNR로 미세조정하며, 라벨이 부족하면 크로스 인코더나 LLM으로 silver 라벨을 만듭니다. 9장의 CLIP도 MNR과 같은 배치 내 대조 학습으로 이미지와 텍스트를 맞춘 모델이고, 주행 장면 검색용 임베딩을 자기 데이터로 맞출 때도 같은 원리를 씁니다.

**끝나면** Studio에서 space를 **Stop** 하세요. 이 노트북은 체크포인트를 저장하지 않으므로 학습한 모델은 커널과 함께 사라집니다. 남기려면 `embedding_model.save("폴더명")`을 학습 직후 실행하세요.